# DATA Preparation

In [ ]:
import pandas as pd
import os
import librosa
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import logging
from pydub import AudioSegment

from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def get_mfcc(y, sr):
    y = librosa.resample(y, orig_sr=sr, target_sr=8000)
    # Tronquer si plus long que 40000 ou compléter si plus court
    y = y[:40000] if len(y) > 40000 else np.pad(y, (0, 40000 - len(y)), 'constant')
    mfcc = librosa.feature.mfcc(y=y, sr=8000, n_mfcc=10, n_fft=2048 , hop_length=1024)
    return mfcc.T.flatten()


def get_pitch(y, sr):
    # Utilisation de pyin pour une meilleure estimation du pitch
    pitches, voiced_flag, voiced_probs = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
    # Filtrer pour obtenir seulement les valeurs où la voix est présente
    voiced_pitches = pitches[voiced_flag & (voiced_probs > 0.9)]
    # Retourner la valeur maximale du pitch détecté
    return np.max(voiced_pitches) if voiced_pitches.size > 0 else 0

def convert_to_wav(file_path):
    # Check if the file is in wav format
    if not file_path.lower().endswith('.wav'):
        logging.info(f"Converting {file_path} to WAV format.")
        # Load file with pydub, which can read many audio formats
        audio_segment = AudioSegment.from_file(file_path)
        # Convert to WAV
        file_path_wav = file_path.rsplit('.', 1)[0] + '.wav'
        audio_segment.export(file_path_wav, format='wav')
        return file_path_wav
    return file_path

def process_audio_file(file_path, gender):
    try:
        file_path = convert_to_wav(file_path)  # Convert file to WAV if needed
        y, sr = librosa.load(file_path, sr=None)
        pitch = get_pitch(y, sr)
        mfcc_features = get_mfcc(y, sr)
        features = np.hstack(([pitch], mfcc_features, [gender]))
        return features
    except Exception as e:
        logging.error(f"Failed to process {file_path}: {e}")
        return None

def process_files(path, gender):
    file_paths = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.wav')]
    tasks = []

    with ProcessPoolExecutor() as executor:
        # Planifier les tâches futures
        futures = [executor.submit(process_audio_file, file_path, gender) for file_path in file_paths]
        
        # Initialiser tqdm pour afficher la progression
        for future in tqdm(futures, total=len(file_paths), desc=f"Processing {gender} files"):
            tasks.append(future.result())

    col = ['pitch'] + ['mfcc' + str(i+1) for i in range(400)] + ['label']
    return pd.DataFrame(tasks, columns=col)

def main():
    gender_paths = {
        'male': "/home/sarah/Desktop/gender-classifier-using-voice/Data_Preprocessing/dataSet/BDD/Male/",
        'female': "/home/sarah/Desktop/gender-classifier-using-voice/Data_Preprocessing/dataSet/BDD/Female/"
    }
    all_features = []

    for gender, path in gender_paths.items():
        gender_features = process_files(path, gender)
        all_features.append(gender_features)

    finaldf = pd.concat(all_features, ignore_index=True)
    finaldf.to_csv('/home/sarah/Desktop/gender-classifier-using-voice/virtuelFFT2048.csv', index=False)

if __name__ == "__main__":
    main()

# verification du nbre de mfcc retournés par get_mfcc

In [2]:
import librosa
import numpy as np

def get_mfcc(y, sr):
    y = librosa.resample(y, orig_sr=sr, target_sr=8000)
    # Tronquer si plus long que 40000 ou compléter si plus court
    y = y[:40000] if len(y) > 40000 else np.pad(y, (0, 40000 - len(y)), 'constant')
    mfcc = librosa.feature.mfcc(y=y, sr=8000, n_mfcc=10, n_fft=512, hop_length=256)
    return mfcc.T.flatten()

# Chargement d'un fichier audio pour tester
file_path = '/home/sarah/Desktop/gender-classifier-using-voice/PFE/BDD/Female/1713293949949.wav'  # Remplacez par le chemin de votre fichier audio
y, sr = librosa.load(file_path, sr=None)

# Calcul des MFCCs
mfcc_features = get_mfcc(y, sr)

print("Nombre de MFCC retournés:", len(mfcc_features))

Nombre de MFCC retournés: 1570


In [137]:
dataset = pd.read_csv("/home/sarah/Desktop/gender-classifier-using-voice/Data_Preprocessing/features.csv")


In [138]:
dataset.describe()